In [1]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import cmocean.cm as cmo

import xarray as xr
import numpy as np
import jax.numpy as jnp
import jaxopt
# import xmitgcm

# from scipy.interpolate import RegularGridInterpolator
# from scipy.interpolate import PchipInterpolator
from scipy.signal.windows import tukey

import sys
sys.path.append('./subroutine/fastjmd95/fastjmd95')
import jmd95numba
sys.path.append('./subroutine')
from isospec_rfft import isospec_rfft

from rfft2 import rfft2, irfft2
from rel_err import rel_err
from isospec_rfft import isospec_rfft
# from detrend_2d import detrend_2d

Here is a snippet of the dimensional version. 

In [ ]:
# dui is the grid spacing in meters, like 2000 m

kx = np.fft.rfftfreq(Eta_filt.shape[1],d=dui)*(2*np.pi)
ky = np.fft.fftfreq(Eta_filt.shape[0],d=dui)*(2*np.pi)

# Then create all the K product as usual. They are just now dimensional [k]=m^-1

In [ ]:
N = 0.00221429 # this is a random number from LLC model output, change it
Kz = Ka*N/f # this is important for z-derivative. It is no longer equal to Ka in dimensional version

In [ ]:
# No dimensional constant for these terms

def CYCLO_TERM__(phi0s__):
    # phi0s__ *= phi0s__*lowpass_pres
    
    phi0s_xx = irfft2(phi0s__*-1*kx**2)
    phi0s_yy = irfft2((phi0s__.T*-1*ky**2).T)
    phi0s_xy = irfft2(phi0s__*-1*jnp.outer(ky,kx))

    # kx_mat, ky_mat = jnp.meshgrid(kx, ky)
    # K2 = (kx_mat**2+ky_mat**2)
    # Kn2 = 1/(K2.at[0,0].set(1e16))
    return -rfft2(2*(phi0s_xx*phi0s_yy-phi0s_xy**2))*Kn2

In [ ]:
# Compare with (37-39), this is those devided by f, since this should have the unit of m/s

def ZETAp1__(phi0s__):
    # phi0s__ *= phi0s__*lowpass_pres
    
    phi0s_xz = irfft2(phi0s__*1j*kx*Kz)
    phi0s_yz = irfft2((phi0s__.T*1j*ky*Kz.T).T)
    term_1 = rfft2(2*(phi0s_xz**2+phi0s_yz**2))

    phi0s_z = irfft2(phi0s__*Kz)
    phi0s_lapz = irfft2(-phi0s__*Kz*K2)
    term_2 = rfft2(2*(phi0s_z*phi0s_lapz))

    phi0s_zz = irfft2(phi0s__*Kz*Kz)
    phi0s_lap = irfft2(-phi0s__*K2)
    term_3 = rfft2((phi0s_zz*phi0s_lap))

    phi0s_x = irfft2(phi0s__*1j*kx)
    phi0s_y = irfft2((phi0s__.T*1j*ky).T)
    phi0s_zzx = irfft2(phi0s__*Kz*Kz*1j*kx)
    phi0s_zzy = irfft2((phi0s__.T*Kz.T*Kz.T*1j*ky).T)
    term_4 = rfft2((phi0s_x*phi0s_zzx+phi0s_y*phi0s_zzy))

    term_5 = rfft2(phi0s_z*phi0s_zz)*Ka
    term_6 = -(rfft2(phi0s_z*phi0s_y).T*1j*ky*Ka.T).T
    term_7 = -rfft2(phi0s_z*phi0s_x)*1j*kx*Ka

    return    f/N**2 *(term_1+term_2+term_3+term_4)*1 \
          +f**2/N**3 *(term_5) *1 \
          +   1/N    *(term_6+term_7) *1 # note mu in the formula is N/f*K

In [ ]:
def ZP1_TERM__(phi0s__):
    return - ZETAp1__(phi0s__)*Kn2*f # here I put the f back 